# Demonstration: LangChain Integration with Hermes 3 (via OpenRouter)

This notebook demonstrates how to build an agentic workflow using the **Hermes 3** model (specifically `nousresearch/hermes-3-llama-3.1-405b:free` on OpenRouter) and **LangChain**.

### Overview of Hermes 3
Hermes 3 is a state-of-the-art open-weights LLM developed by Nous Research. It excels at agentic tasks, function calling, complex reasoning, and structured output. In this notebook, we use the OpenRouter API to query the 405B parameter version of Hermes 3.

### Table of Contents
1. **Setup & Installation**: Installing necessary LangChain packages.
2. **Environment Setup**: Safely prompt for the OpenRouter API key.
3. **Defining Custom Tools**: Creating custom LangChain tools using the `@tool` decorator.
4. **Model Initialization**: Setting up the Hermes 3 model with tools bound to it.
5. **Single Tool Execution**: Showing the raw tool call generation for a simple query.
6. **Agentic Reasoning & Execution**: A complete decision-making and execution workflow.

## 1. Setup & Installation

First, we install the required packages. We need `langchain-openai` to connect to OpenRouter (using the OpenAI-compatible API) and `langchain-core` for core abstractions like tools and messaging.

In [2]:
# Install the necessary packages for LangChain OpenAI integration and core features.
# Using %pip ensures installation within the correct environment context.
%pip install -U langchain-openai langchain-core

Note: you may need to restart the kernel to use updated packages.


## 2. Environment Setup

To access OpenRouter, you must set the `OPENROUTER_API_KEY` environment variable. We use `getpass` to securely prompt for the key without displaying it on the screen.

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

# Check if the environment variable is already set; if not, prompt securely.
if not os.environ.get("OPENROUTER_API_KEY"):
    print("Please enter your OpenRouter API Key:")
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass()

print("OpenRouter API Key has been configured successfully.")

Please enter your OpenRouter API Key:
OpenRouter API Key has been configured successfully.


## 3. Defining Custom Tools

We define two custom tools using LangChain's `@tool` decorator. Detailed docstrings and type hints are crucial because LangChain extracts them to build the JSON schema that is sent to the LLM. The model relies on this schema to decide which tool to invoke.

*   `get_mock_stock_price`: Fetches stock price information.
*   `calculate_roi`: Calculates Return on Investment (ROI) based on investment numbers.

In [3]:
from langchain_core.tools import tool

@tool
def get_mock_stock_price(ticker: str) -> float:
    """
    Fetch the mock current stock price for a given ticker symbol.
    
    Args:
        ticker (str): The stock ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOG', 'AMZN', 'TSLA').
        
    Returns:
        float: The mock current price of the stock.
    """
    # A static dictionary representing our mock stock database
    mock_prices = {
        "AAPL": 175.50,
        "MSFT": 420.20,
        "GOOG": 150.75,
        "AMZN": 180.10,
        "TSLA": 170.30
    }
    
    # Standardize the ticker format to uppercase
    ticker_upper = ticker.strip().upper()
    price = mock_prices.get(ticker_upper, 100.0)  # Default price if ticker is not found
    print(f"[Tool Execution] get_mock_stock_price for '{ticker_upper}' -> ${price}")
    return price


@tool
def calculate_roi(initial_investment: float, current_value: float) -> float:
    """
    Calculate the Return on Investment (ROI) percentage based on the initial investment and current value.
    
    Args:
        initial_investment (float): The initial amount of money invested.
        current_value (float): The current or ending value of the investment.
        
    Returns:
        float: The calculated Return on Investment (ROI) as a percentage.
    """
    if initial_investment <= 0:
        raise ValueError("Initial investment must be greater than zero.")
        
    roi = ((current_value - initial_investment) / initial_investment) * 100
    print(f"[Tool Execution] calculate_roi (Initial: ${initial_investment}, Current: ${current_value}) -> {roi:.2f}%")
    return round(roi, 2)

## 4. Model Initialization

We use `ChatOpenAI` pointing to the OpenRouter endpoint. We set the model to `nousresearch/hermes-3-llama-3.1-405b:free` and set `temperature=0.0` to ensure stable, deterministic responses. Then, we bind our custom tools using `.bind_tools()`.

In [1]:
from langchain_openai import ChatOpenAI

# Read OpenRouter API key from the environment
api_key = os.environ.get("OPENROUTER_API_KEY")

# Initialize ChatOpenAI for the Hermes 3 model via OpenRouter API
llm = ChatOpenAI(
    # model="nousresearch/hermes-3-llama-3.1-405b"
    # model="nousresearch/hermes-3-llama-3.1-405b:free",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.0
)

# Bind our custom tools to the LLM instance
tools = [get_mock_stock_price, calculate_roi]
llm_with_tools = llm.bind_tools(tools)

print("Hermes 3 model initialized and tools bound successfully.")

NameError: name 'os' is not defined

## 5. Single Tool Execution

Let's test the model's ability to trigger a tool call for a query that requests stock price info. We will print the raw output message to inspect the structure of the tool call payload generated by Hermes 3.

In [5]:
from langchain_core.messages import HumanMessage

# Formulate a query for stock price
query_single = "What is the current stock price of Microsoft (MSFT)?"
print(f"User Query: {query_single}\n")

# Invoke the tool-bound model
response_single = llm_with_tools.invoke([HumanMessage(content=query_single)])

print("--- Raw LLM Response ---")
print(response_single)

print("\n--- Extracted Tool Calls ---")
if response_single.tool_calls:
    for tool_call in response_single.tool_calls:
        print(f"Tool Name: {tool_call['name']}")
        print(f"Arguments: {tool_call['args']}")
        print(f"Tool Call ID: {tool_call['id']}")
else:
    print("No tool call was generated. Ensure your OpenRouter API key is set and valid.")

User Query: What is the current stock price of Microsoft (MSFT)?



NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found that support tool use. Try disabling "get_mock_stock_price". To learn more about provider routing, visit: https://openrouter.ai/docs/guides/routing/provider-selection', 'code': 404}}

## 6. Agentic Reasoning & Decision Making

In this section, we build a simple, complete execution loop. The agent must decide which tool to call, execute the tool locally, feed the results back to the model, and allow the model to construct a final user-friendly response.

In [ ]:
from langchain_core.messages import ToolMessage

# Create a mapping of tool names to their corresponding tool functions
tool_map = {
    "get_mock_stock_price": get_mock_stock_price,
    "calculate_roi": calculate_roi
}

# Query that requires ROI calculation
query_complex = "I bought some stocks for $5,000 and sold them for $7,200. Calculate my return on investment."
print(f"User Query: {query_complex}\n")

# Ask Hermes 3 to decide the next step
initial_response = llm_with_tools.invoke([HumanMessage(content=query_complex)])

if initial_response.tool_calls:
    # Process each tool call suggested by the model
    messages = [HumanMessage(content=query_complex), initial_response]
    
    for tool_call in initial_response.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]
        tool_id = tool_call["id"]
        
        print(f"Model selected tool: '{name}'")
        print(f"Arguments: {args}")
        
        if name in tool_map:
            # Execute the tool with the provided arguments
            tool_func = tool_map[name]
            result = tool_func.invoke(args)
            
            print(f"Tool Result: {result}\n")
            
            # Append the result as a ToolMessage to the conversation history
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))
        else:
            print(f"Error: Tool '{name}' is not defined in the tool map.")
            
    # Get the final response from the model integrating the tool output
    print("Asking LLM to compile final response...")
    final_response = llm.invoke(messages)
    
    print("\n--- Final Agent Response ---")
    print(final_response.content)
else:
    print("The model responded directly without calling any tools:")
    print(initial_response.content)